In [34]:
import os
import sys
import argparse
import random
import re
import unicodedata
from pathlib import Path

import pandas as pd

from sacrebleu import BLEU
from rouge_score import rouge_scorer

# Locate repo root so imports work regardless of launch dir
repo_root = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "modules").exists():
        repo_root = candidate.resolve()
        break

if repo_root is None:
    raise RuntimeError("Could not locate project root containing modules/")

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

os.chdir(repo_root)

from modules.main import generate_meal_plan
from modules.Validator import Validator



In [39]:
# Configuration
NUM_SAMPLES = 10  # set how many recipes to sample
DATASET_PATH = repo_root / "full_dataset.csv"
RANDOM_SEED = 42
SHOW_MEAL_PLAN = False  # set True to print generated meal plans



In [36]:
# Helpers for parsing meal plan sections and computing metrics
bleu_scorer = BLEU()
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

MEAL_SECTION_PATTERN = re.compile(
    r"""
    (?m)
    (^|\n)\s*#{0,3}\s*
    (BREAKFAST|LUNCH|DINNER)
    \s*[:–\-]*\s*
    \n
    (.*?)
    (?=
        \n\s*#{0,3}\s*(?:BREAKFAST|LUNCH|DINNER)\b
        |\Z
    )
    """,
    re.IGNORECASE | re.DOTALL | re.VERBOSE,
)

SECTION_HEADER_RE = re.compile(r"^\s*([1-6])\.\s*(.*)$", re.MULTILINE)

def clean_text(text: str) -> str:
    """Normalize unicode, strip odd chars, and collapse whitespace."""
    if not isinstance(text, str):
        text = str(text)
    normalized = unicodedata.normalize("NFKD", text)
    ascii_text = normalized.encode("ascii", "ignore").decode("ascii")
    ascii_text = ascii_text.replace("\r", "")
    ascii_text = re.sub(r"[ \t]+\n", "\n", ascii_text)
    ascii_text = re.sub(r"\n{2,}", "\n\n", ascii_text)
    return ascii_text.strip()

_validator_for_parse = None

def get_validator_for_parse():
    global _validator_for_parse
    if _validator_for_parse is None:
        dummy_args = argparse.Namespace(
            val_model="gpt-oss:20b",
            verbose=False,
            check_budget=False,
            allergens=None,
            ingredients="",
        )
        _validator_for_parse = Validator(dummy_args)
    return _validator_for_parse

def split_meals(meal_plan: str):
    """Use Validator.extract_recipes first, then fall back to tolerant parsing."""
    meals = {}
    try:
        validator = get_validator_for_parse()
        meals = validator.extract_recipes(meal_plan)
        meals = {k: v for k, v in meals.items() if v}
    except Exception as e:
        print(f"Validator extract_recipes failed: {e}")
        meals = {}

    if meals:
        return meals

    # Strategy 1: simple split on headers
    parts = re.split(r"##\s*(BREAKFAST|LUNCH|DINNER)\s*", meal_plan, flags=re.IGNORECASE)
    if len(parts) > 1:
        for i in range(1, len(parts), 2):
            header = parts[i].lower()
            body = parts[i + 1] if i + 1 < len(parts) else ""
            meals[header] = body.strip()

    if meals:
        return meals

    # Strategy 2: regex matcher
    matches = MEAL_SECTION_PATTERN.findall(meal_plan)
    for _, header, body in matches:
        meals[header.lower()] = body.strip()

    if meals:
        return meals

    # Strategy 3: last-resort greedy search
    for key in ["BREAKFAST", "LUNCH", "DINNER"]:
        pattern = rf"{key}(.*?)(?=BREAKFAST|LUNCH|DINNER|\Z)"
        m = re.search(pattern, meal_plan, flags=re.IGNORECASE | re.DOTALL)
        if m:
            meals[key.lower()] = m.group(1).strip()
    return meals

def extract_sections_1_3_4(meal_body: str):
    """Extract sections 1 (title), 3 (ingredients), 4 (instructions) using numbered headers only."""
    matches = list(SECTION_HEADER_RE.finditer(meal_body))
    sections = {}

    if matches:
        for idx, m in enumerate(matches):
            num = int(m.group(1))
            start = m.end()
            end = matches[idx + 1].start() if idx + 1 < len(matches) else len(meal_body)
            sections[num] = meal_body[start:end].strip()
    else:
        # Fallback: loose numbered capture
        for num in [1, 3, 4]:
            pat = rf"{num}\.\s*(.*?)(?=\n\d\.\s|\Z)"
            m = re.search(pat, meal_body, flags=re.DOTALL)
            if m:
                sections[num] = m.group(1).strip()

    title = sections.get(1, "")
    ingredients = sections.get(3, "")
    instructions = sections.get(4, "")
    return title, ingredients, instructions

def combine_sections(parts):
    return "\n".join([p for p in parts if p]).strip()

def compute_bleu_rouge(candidate: str, reference: str):
    bleu_score = bleu_scorer.corpus_score([candidate], [[reference]]).score / 100.0
    rouge_score = rouge.score(reference, candidate)["rougeL"].fmeasure
    return bleu_score, rouge_score

def evaluate_plan(meal_plan: str, reference_text: str):
    ref_clean = clean_text(reference_text)
    plan_clean = clean_text(meal_plan)

    meals = split_meals(plan_clean)
    per_meal = []
    for meal_name in ["breakfast", "lunch", "dinner"]:
        body = meals.get(meal_name)
        if not body:
            continue
        title, ingredients, instructions = extract_sections_1_3_4(body)
        combined = combine_sections([title, ingredients, instructions])
        if not combined:
            continue
        try:
            bleu_score, rouge_score = compute_bleu_rouge(combined, ref_clean)
            per_meal.append({"meal": meal_name, "bleu": bleu_score, "rougel": rouge_score})
        except Exception as e:
            print(f"Metric error for {meal_name}: {e}")

    if per_meal:
        avg_bleu = sum(m["bleu"] for m in per_meal) / len(per_meal)
        avg_rouge = sum(m["rougel"] for m in per_meal) / len(per_meal)
        return {"bleu": avg_bleu, "rougel": avg_rouge, "per_meal": per_meal}

    # fallback: compare whole plan if sections failed to parse
    try:
        bleu_score, rouge_score = compute_bleu_rouge(plan_clean, ref_clean)
        print("it fkin failed")
        return {"bleu": bleu_score, "rougel": rouge_score, "per_meal": []}
    except Exception as e:
        print(f"Metric error (fallback): {e}")
        return {"bleu": 0.0, "rougel": 0.0, "per_meal": []}



In [40]:
# Load dataset and sample recipes
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=["title", "ingredients", "directions"])
sample_count = min(NUM_SAMPLES, len(df))
random.seed(RANDOM_SEED)
df_sample = df.sample(n=sample_count, random_state=RANDOM_SEED)

print(f"Sampling {sample_count} recipes from {DATASET_PATH}")



Sampling 10 recipes from /Users/selenahuang/school/llm-recipe-generation/full_dataset.csv


In [6]:
results = []
log_path = repo_root / "testing.txt"
orig_path = repo_root / "original.txt"

for i, row in enumerate(df_sample.itertuples(index=False), start=1):
    ingredients_input = str(row.ingredients)
    print(f"\n=== Sample {i} ingredients ===\n{ingredients_input}\n")

    args = argparse.Namespace(
        model="gpt-oss:20b",
        val_model="gpt-oss:20b",
        ingredients=ingredients_input,
        max_iterations=1,
        verbose=False,
        allergens=None,
        budget=None,
        calories=None,
        check_budget=False,
    )

    generated = generate_meal_plan(args)
    if isinstance(generated, tuple) and len(generated) >= 1:
        meal_plan_raw = generated[0]
    else:
        meal_plan_raw = str(generated)

    meal_plan = clean_text(meal_plan_raw)

    if SHOW_MEAL_PLAN:
        print(meal_plan)

    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"Generated: {meal_plan}\n\n")

    reference_text_raw = "\n".join([str(row.title), str(row.ingredients), str(row.directions)])
    reference_text = clean_text(reference_text_raw)
    with open(orig_path, "a", encoding="utf-8") as f:
        f.write(f"Original: {reference_text}\n\n")

    metrics = evaluate_plan(meal_plan, reference_text)
    results.append({"bleu": metrics["bleu"], "rougel": metrics["rougel"]})
    print(f"Metrics: {results[-1]}")

print("\nAll results:", results)
results




=== Sample 1 ingredients ===
["12 cup butter", "1 cup brown sugar", "6 cups crisp rice cereal", "1 cup coconut", "12 cup chopped nuts", "12 gallon vanilla ice cream, softened", "16 ounces strawberries, cleaned, stemmed, & sliced", "1 pint strawberry, cleaned and stemmed", "13 cup white sugar", "1 teaspoon vanilla"]

evaluating plan
Metrics: {'bleu': 0.0, 'rougel': 0.0}

All results: [{'bleu': 0.0, 'rougel': 0.0}]


[{'bleu': 0.0, 'rougel': 0.0}]

In [29]:
# Compare generated vs original files
import ast

def parse_entries(path: Path, prefix: str):
    text = path.read_text(encoding="utf-8") if path.exists() else ""
    parts = []
    for chunk in text.split(f"{prefix}:"):
        chunk = chunk.strip()
        if not chunk:
            continue
        parts.append(chunk)
    return parts

def prep_reference(original_chunk: str):
    lines = [l for l in original_chunk.splitlines() if l.strip()]
    title = lines[0] if len(lines) >= 1 else ""
    ingredients_raw = lines[1] if len(lines) >= 2 else ""
    instructions = lines[2] if len(lines) >= 3 else ""
    ingredients_text = ingredients_raw
    try:
        parsed = ast.literal_eval(ingredients_raw)
        if isinstance(parsed, (list, tuple)):
            ingredients_text = "; ".join(str(x) for x in parsed)
    except Exception:
        pass
    return clean_text("\n".join([title, ingredients_text, instructions]))

def prep_meal_plan(generated_chunk: str):
    return clean_text(generated_chunk)

generated_chunks = parse_entries(log_path, "Generated")
original_chunks = parse_entries(orig_path, "Original")

pair_count = min(len(generated_chunks), len(original_chunks))
comparison_results = []

for idx in range(pair_count):
    ref_text = prep_reference(original_chunks[idx])
    meal_plan_text = prep_meal_plan(generated_chunks[idx])
    metrics = evaluate_plan(meal_plan_text, ref_text)
    comparison_results.append(metrics)
    print(f"Pair {idx+1}: {metrics}")

print("\nComparison results:", comparison_results)
comparison_results



Meals: {'breakfast': '1. Title\nCrispy Coconut Strawberry Breakfast Bowl\n\n2. Serving size\n1 bowl\n\n3. Ingredients and amounts\n180 g crisp rice cereal\n40 g coconut\n454 g strawberries\n5 ml vanilla\n\n4. Instructions\n- Place the crisp rice cereal into a large bowl.\n- Sprinkle the coconut over the cereal.\n- Add the sliced strawberries evenly on top.\n- Drizzle the vanilla through the mixture.\n- Gently stir to combine flavors before serving.\n\n5. Nutritional information', 'lunch': '1. Title\nButter Brown Sugar Nutty Strawberry Plate\n\n2. Serving size\n1 plate\n\n3. Ingredients and amounts\n2724 g butter\n200 g brown sugar\n720 g chopped nuts\n473 g strawberries\n\n4. Instructions\n- Melt the butter in a saucepan over low heat.\n- Stir in the brown sugar until fully dissolved.\n- Toss the chopped nuts in the warm buttersugar mixture until they are well coated.\n- Arrange the coated nuts on a plate and top with sliced strawberries.\n- Serve immediately.\n\n5. Nutritional informa

[{'bleu': 0.005851086514995313,
  'rougel': 0.17344711723391085,
  'per_meal': [{'meal': 'breakfast',
    'bleu': 0.0028932933034777316,
    'rougel': 0.15165876777251186},
   {'meal': 'lunch',
    'bleu': 0.004812595545752608,
    'rougel': 0.2162162162162162},
   {'meal': 'dinner',
    'bleu': 0.009847370695755602,
    'rougel': 0.1524663677130045}]}]

In [45]:
# Generate and evaluate NUM_SAMPLES meal plans directly (no file reads)
import pprint

results = []
for i, row in enumerate(df_sample.itertuples(index=False), start=4):
    ingredients_input = str(row.ingredients)
    print(f"\n=== Sample {i} ingredients ===\n{ingredients_input}\n")

    args = argparse.Namespace(
        model="gpt-oss:20b",
        val_model="gpt-oss:20b",
        ingredients=ingredients_input,
        max_iterations=3,
        verbose=False,
        allergens=None,
        budget=None,
        calories=None,
        check_budget=False,
    )

    generated = generate_meal_plan(args)
    meal_plan_raw = generated[0] if isinstance(generated, tuple) and len(generated) >= 1 else str(generated)
    meal_plan = clean_text(meal_plan_raw)

    if SHOW_MEAL_PLAN:
        print(meal_plan)

    reference_text = clean_text("\n".join([str(row.title), str(row.ingredients), str(row.directions)]))
    metrics = evaluate_plan(meal_plan, reference_text)
    results.append({
        "sample": i,
        "bleu": metrics.get("bleu", 0.0),
        "rougel": metrics.get("rougel", 0.0),
        "per_meal": metrics.get("per_meal", []),
    })
    print(f"Sample {i} metrics:")
    pprint.pp(results[-1])

print("\nAll sample metrics:")
pprint.pp(results)
results




=== Sample 4 ingredients ===
["12 cup butter", "1 cup brown sugar", "6 cups crisp rice cereal", "1 cup coconut", "12 cup chopped nuts", "12 gallon vanilla ice cream, softened", "16 ounces strawberries, cleaned, stemmed, & sliced", "1 pint strawberry, cleaned and stemmed", "13 cup white sugar", "1 teaspoon vanilla"]

Sample 4 metrics:
{'sample': 4,
 'bleu': 0.0,
 'rougel': 0.0375,
 'per_meal': [{'meal': 'breakfast', 'bleu': 0.0, 'rougel': 0.0375},
              {'meal': 'lunch', 'bleu': 0.0, 'rougel': 0.0375},
              {'meal': 'dinner', 'bleu': 0.0, 'rougel': 0.0375}]}

=== Sample 5 ingredients ===
["1 2/3 cups pinto beans", "1 bay leaf", "1 9/16 pounds pork ribs smoked", "1/2 cup celery root cleaned", "1 parsley root cleaned", "1 carrot trimmed", "1 red pepper cleaned", "2 chili peppers small, cleaned", "2 tomatoes seedless", "1 tablespoon olive oil", "13/16 cup onions chopped", "2 cloves garlic chopped", "4 1/4 cups water", "1/4 cup tomato puree", "1 teaspoon coarse sea salt"]


[{'sample': 4,
  'bleu': 0.0,
  'rougel': 0.0375,
  'per_meal': [{'meal': 'breakfast', 'bleu': 0.0, 'rougel': 0.0375},
   {'meal': 'lunch', 'bleu': 0.0, 'rougel': 0.0375},
   {'meal': 'dinner', 'bleu': 0.0, 'rougel': 0.0375}]},
 {'sample': 5,
  'bleu': 0.010777939839577692,
  'rougel': 0.25688073394495414,
  'per_meal': [{'meal': 'breakfast',
    'bleu': 0.010777939839577692,
    'rougel': 0.25688073394495414},
   {'meal': 'lunch',
    'bleu': 0.010777939839577692,
    'rougel': 0.25688073394495414},
   {'meal': 'dinner',
    'bleu': 0.010777939839577692,
    'rougel': 0.25688073394495414}]},
 {'sample': 6,
  'bleu': 6.290716977751244e-05,
  'rougel': 0.05405405405405406,
  'per_meal': [{'meal': 'breakfast',
    'bleu': 6.290716977751244e-05,
    'rougel': 0.05405405405405406},
   {'meal': 'lunch',
    'bleu': 6.290716977751244e-05,
    'rougel': 0.05405405405405406},
   {'meal': 'dinner',
    'bleu': 6.290716977751244e-05,
    'rougel': 0.05405405405405406}]},
 {'sample': 7,
  'bleu':

In [47]:
print(results)

[{'sample': 4, 'bleu': 0.0, 'rougel': 0.0375, 'per_meal': [{'meal': 'breakfast', 'bleu': 0.0, 'rougel': 0.0375}, {'meal': 'lunch', 'bleu': 0.0, 'rougel': 0.0375}, {'meal': 'dinner', 'bleu': 0.0, 'rougel': 0.0375}]}, {'sample': 5, 'bleu': 0.010777939839577692, 'rougel': 0.25688073394495414, 'per_meal': [{'meal': 'breakfast', 'bleu': 0.010777939839577692, 'rougel': 0.25688073394495414}, {'meal': 'lunch', 'bleu': 0.010777939839577692, 'rougel': 0.25688073394495414}, {'meal': 'dinner', 'bleu': 0.010777939839577692, 'rougel': 0.25688073394495414}]}, {'sample': 6, 'bleu': 6.290716977751244e-05, 'rougel': 0.05405405405405406, 'per_meal': [{'meal': 'breakfast', 'bleu': 6.290716977751244e-05, 'rougel': 0.05405405405405406}, {'meal': 'lunch', 'bleu': 6.290716977751244e-05, 'rougel': 0.05405405405405406}, {'meal': 'dinner', 'bleu': 6.290716977751244e-05, 'rougel': 0.05405405405405406}]}, {'sample': 7, 'bleu': 5.3798423097029296e-06, 'rougel': 0.06666666666666667, 'per_meal': [{'meal': 'breakfast'

In [ ]:
import matplotlib.pyplot as plt

# Plot BLEU and ROUGE-L for every sample in results
run_results = locals().get("results", [])
if not run_results:
    print("No results to plot.")
else:
    samples = [entry.get("sample", idx + 1) for idx, entry in enumerate(run_results)]
    bleu_scores = [entry.get("bleu", 0.0) for entry in run_results]
    rouge_scores = [entry.get("rougel", 0.0) for entry in run_results]

    x = range(len(samples))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars_bleu = ax.bar([i - width / 2 for i in x], bleu_scores, width, label="BLEU", color="skyblue")
    bars_rouge = ax.bar([i + width / 2 for i in x], rouge_scores, width, label="ROUGE-L", color="salmon")

    ax.set_xticks(list(x))
    ax.set_xticklabels([str(s) for s in samples])
    ax.set_xlabel("Sample")
    ax.set_ylabel("Score")
    ax.set_title("Metrics per Sample")
    ax.legend()

    for bar in list(bars_bleu) + list(bars_rouge):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.002, f"{height:.3f}", ha="center")

    upper = max(bleu_scores + rouge_scores + [0]) + 0.05
    ax.set_ylim(0, upper)
    plt.show()


ModuleNotFoundError: No module named 'matplotlib'